In [20]:
# Limpiar conexiones previas y archivos bloqueados
import os
import time

# Eliminar archivos bloqueados de DuckDB
try:
    if os.path.exists('/workspaces/base-de-datos/analytics.duckdb.wal'):
        os.remove('/workspaces/base-de-datos/analytics.duckdb.wal')
        print("Archivo .wal eliminado")
    if os.path.exists('/workspaces/base-de-datos/analytics.duckdb'):
        os.remove('/workspaces/base-de-datos/analytics.duckdb')
        print("Archivo .duckdb eliminado")
except Exception as e:
    print(f"Nota: {e}")

time.sleep(1)  # Esperar un momento para liberar bloqueos


Archivo .duckdb eliminado


In [21]:
pip install duckdb openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:
import duckdb

In [23]:
pip install pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
import pandas as pd

In [36]:
try:
    conn = duckdb.connect('analytics.duckdb')
    print("Conexión establecida exitosamente")
except Exception as e:
    print(f"Error al conectar: {e}")
    print("Intenta ejecutar la primera celda para limpiar los archivos bloqueados")


Conexión establecida exitosamente


In [26]:
datos= pd.read_excel("/workspaces/base-de-datos/data/ej base datos 2.xlsx")

In [27]:
datos

,Fecha,Precio,Activo,Agregacion
0,2021-01-01,100,Precio,Daily
1,2021-01-01,100,Precio,Daily
2,2021-01-01,100,Precio,Daily
3,2021-01-01,100,Precio,Daily
4,2021-01-01,100,Precio,Daily
5,2021-01-01,100,Precio,Daily
6,2021-01-01,100,Precio,Daily
7,2021-01-01,100,Precio,Daily
8,2021-01-01,100,Precio,Daily
9,2021-01-01,100,Precio,Daily


In [28]:
df_fb = datos[datos["Activo"] == "FB"]

In [29]:
conn.execute("CREATE TABLE datos_fb AS SELECT * FROM df_fb")

In [30]:

conn.execute("CREATE SCHEMA IF NOT EXISTS bronze")
conn.execute("CREATE SCHEMA IF NOT EXISTS silver")
conn.execute("CREATE SCHEMA IF NOT EXISTS gold")


In [39]:
# Cerrar la conexión correctamente
conn.close()
print("Conexión cerrada")


Conexión cerrada


In [32]:
pip install yfinance


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [33]:
import yfinance as yf

In [34]:
# Descargar datos históricos
df = yf.download("F", start="2020-01-01")

# reset index para que la fecha sea columna
df = df.reset_index()
df.columns = ["fecha", "close", "high", "low", "open", "volume"]
df = df.astype(str)

print(df.head())


[*********************100%***********************]  1 of 1 completed

        fecha              close               high                 low  \
0  2020-01-02  6.952615737915039  6.952615737915039   6.782859362687308   
1  2020-01-03  6.797622203826904  6.915713247352708   6.753337710565657   
2  2020-01-06  6.760717868804932   6.76809873422194   6.686911326269049   
3  2020-01-07  6.827144622802734  6.827144622802734   6.731195478773647   
4  2020-01-08  6.827144622802734  6.864048248025938  6.7680991039968506   

                open    volume  
0  6.856666604765109  43425700  
1  6.871429457969603  45040800  
2  6.716434084059016  43372300  
3  6.790240997579531  44984100  
4   6.81238289116221  45994900  


In [37]:
conn.execute("""
CREATE OR REPLACE TABLE bronze.ford_raw AS
SELECT * FROM df
""")

In [38]:
conn.execute("""
CREATE OR REPLACE TABLE silver.ford_prices AS
SELECT
CAST(fecha AS DATE) AS fecha,
CAST(close AS DOUBLE) AS close,
CAST(high AS DOUBLE) AS high,
CAST(low AS DOUBLE) AS low,
CAST(open AS DOUBLE) AS open,
CAST(volume AS BIGINT) AS volume
FROM bronze.ford_raw;
""")